# Install & Imports

In [ ]:
!pip install pydeck osmnx --quiet

import os
import numpy as np
import pandas as pd
import joblib
import osmnx as ox
import pydeck as pdk
from scipy.spatial import cKDTree
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/SU Works/CPSC_5310_Project'
DATA_PATH    = os.path.join(PROJECT_ROOT, 'saved_data')
MODEL_PATH   = os.path.join(PROJECT_ROOT, 'saved_models')
OUTPUT_PATH  = os.path.join(PROJECT_ROOT, 'notebooks', 'final_heatmap')
os.makedirs(OUTPUT_PATH, exist_ok=True)

print('Ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 5.5 MB/s eta 0:00:00
Mounted at /content/drive
Ready


Load Models & Data

In [ ]:
xgb_model    = joblib.load(os.path.join(MODEL_PATH, 'xgb_best_model.pkl'))
preprocessor = joblib.load(os.path.join(MODEL_PATH, 'xgb_preprocessor.pkl'))
kmeans       = joblib.load(os.path.join(MODEL_PATH, 'kmeans_model_k150.pkl'))
scaler       = joblib.load(os.path.join(MODEL_PATH, 'spatial_scaler.pkl'))
test_df      = pd.read_parquet(os.path.join(DATA_PATH, 'baseline_testing.parquet'))

centers     = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame({
    'cluster_id': range(150),
    'lat':        centers[:, 0],
    'lon':        centers[:, 1],
})

available = sorted(test_df['timestamp'].unique())
print(f'Test period : {test_df["timestamp"].min()} → {test_df["timestamp"].max()}')
print(f'Centroids   : {len(centroid_df)}')

/usr/lib/python3.12/pickle.py:1760: UserWarning: [02:39:09] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [02:39:09] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [02:39:09] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


Test period : 2016-03-19 23:00:00 → 2016-03-31 22:00:00
Centroids   : 150


Fetch NYC Road Network from OSM

Downloads the drivable road network for the NYC bounding box. osmnx caches results locally

In [ ]:
print('Fetching OSM road network — Manhattan …')

G = ox.graph_from_place(
    'Manhattan, New York City, New York, USA',
    network_type='drive',
    simplify=True,
    retain_all=False,
)

edges_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True).reset_index()
print(f'Road segments : {len(edges_gdf):,}')

def geom_midpoint(geom):
    coords = list(geom.coords)
    mid    = coords[len(coords) // 2]
    return mid[1], mid[0]   # lat, lon

edges_gdf[['mid_lat', 'mid_lon']] = edges_gdf['geometry'].apply(
    lambda g: pd.Series(geom_midpoint(g))
)

def geom_to_path(geom):
    return [[c[0], c[1]] for c in geom.coords]

edges_gdf['path'] = edges_gdf['geometry'].apply(geom_to_path)

Fetching OSM road network — Manhattan …
Road segments : 9,916


# Neon Color Palette & Helpers

Each revenue class gets:
- A **core color** (bright, opaque) for the thin center line
- A **bloom color** (same hue, ~15% alpha) for the wide glow underneath
- A **width** — higher activity = thicker roads

In [ ]:
TARGET_COL      = 'revenue_class'
COLS_TO_EXCLUDE = [TARGET_COL, 'timestamp', 'hour', 'day_of_week']

# Vivid neon core colors — [R, G, B, A]
CLASS_CORE = {
    1: [80,  130, 255, 210],   # electric blue  — Quiet
    2: [0,   230, 200, 220],   # neon cyan-teal — Steady
    3: [100, 255, 100, 230],   # neon green     — Busy
    4: [255, 210, 0,   240],   # neon amber     — High Value
    5: [255, 60,  80,  255],   # neon red-pink  — Surge
}

# Same hue, wide + very transparent → glow bleed effect
CLASS_BLOOM = {
    1: [80,  130, 255, 35],
    2: [0,   230, 200, 40],
    3: [100, 255, 100, 45],
    4: [255, 210, 0,   50],
    5: [255, 60,  80,  60],
}

CLASS_CORE_WIDTH  = {1: 2,  2: 2.5, 3: 3.5, 4: 5,  5: 7  }   # pixels
CLASS_BLOOM_WIDTH = {1: 8, 2: 12,  3: 18,  4: 24, 5: 32 }   # pixels (6–8× core)

CLASS_LABEL = {1: 'Quiet', 2: 'Steady', 3: 'Busy', 4: 'High Value', 5: 'Surge'}


def predict_hour(timestamp: str) -> pd.DataFrame:
    """Return per-cluster revenue predictions for a given timestamp."""
    ts   = pd.Timestamp(timestamp)
    h_df = test_df[test_df['timestamp'] == ts].copy()

    if len(h_df) == 0:
        test_df['_ts_floor'] = pd.to_datetime(test_df['timestamp']).dt.floor('h')
        h_df = test_df[test_df['_ts_floor'] == ts].copy()
        test_df.drop(columns=['_ts_floor'], inplace=True)

    if len(h_df) == 0:
        raise ValueError(
            f'No data for {timestamp}. '
            f'Available: {test_df["timestamp"].min()} → {test_df["timestamp"].max()}'
        )

    X_enc = preprocessor.transform(h_df.drop(columns=COLS_TO_EXCLUDE))
    preds = xgb_model.predict(X_enc) + 1    # 0-indexed → 1–5

    pred_df = pd.DataFrame({
        'cluster_id':    h_df['pickup_cluster'].values,
        'revenue_class': preds.astype(int),
        'demand_count':  h_df['demand_count'].values,
    })

    merged = centroid_df.merge(pred_df, on='cluster_id', how='left')
    merged['revenue_class'] = merged['revenue_class'].fillna(1).astype(int)
    return merged


def build_road_layers(cluster_preds: pd.DataFrame):
    """
    Assigns each road segment a revenue class by blending the 3 nearest
    cluster centroids, weighted by inverse distance.
    Smoother transitions than nearest-only — no hard Voronoi boundaries.
    Returns (bloom_layer, core_layer).
    """
    tree = cKDTree(cluster_preds[['lat', 'lon']].values)

    # k=3 nearest centroids per road segment midpoint
    dists, idxs = tree.query(edges_gdf[['mid_lat', 'mid_lon']].values, k=3)

    # Inverse distance weights — closer centroid = stronger influence
    weights = 1.0 / (dists + 1e-9)
    weights = weights / weights.sum(axis=1, keepdims=True)   # normalize rows

    # Weighted blend of revenue class → round to nearest int, clip 1–5
    classes = cluster_preds['revenue_class'].values[idxs]    # shape (N, 3)
    blended = (classes * weights).sum(axis=1)

    road_df = edges_gdf[['path']].copy()
    road_df['revenue_class'] = np.clip(np.round(blended), 1, 5).astype(int)
    road_df['core_color']    = road_df['revenue_class'].map(CLASS_CORE)
    road_df['bloom_color']   = road_df['revenue_class'].map(CLASS_BLOOM)
    road_df['core_width']    = road_df['revenue_class'].map(CLASS_CORE_WIDTH)
    road_df['bloom_width']   = road_df['revenue_class'].map(CLASS_BLOOM_WIDTH)
    road_df['label']         = road_df['revenue_class'].map(CLASS_LABEL)

    bloom_layer = pdk.Layer(
        'PathLayer',
        data=road_df,
        get_path='path',
        get_color='bloom_color',
        get_width='bloom_width',
        width_min_pixels=1,
        width_scale=1,
        pickable=False,
        rounded=True,
        billboard=False,
    )

    core_layer = pdk.Layer(
        'PathLayer',
        data=road_df,
        get_path='path',
        get_color='core_color',
        get_width='core_width',
        width_min_pixels=1,
        width_scale=1,
        pickable=True,
        auto_highlight=True,
        rounded=True,
        billboard=False,
    )

    return bloom_layer, core_layer

# Timeline for 24 Hours

In [ ]:
import json as _json

TIMELINE_HOURS = [str(t) for t in available[:24]]
print(f'Precomputing {len(TIMELINE_HOURS)} hours …')

road_paths = edges_gdf['path'].tolist()
print(f'Road segments : {len(road_paths):,}')

timeline_data = {}
for ts_str in TIMELINE_HOURS:
    preds = predict_hour(ts_str)
    tree  = cKDTree(preds[['lat', 'lon']].values)
    dists, idxs = tree.query(edges_gdf[['mid_lat', 'mid_lon']].values, k=3)
    weights = 1.0 / (dists + 1e-9)
    weights = weights / weights.sum(axis=1, keepdims=True)
    classes = preds['revenue_class'].values[idxs]
    blended = np.clip(np.round((classes * weights).sum(axis=1)), 1, 5).astype(int)
    timeline_data[ts_str] = {
        'core':  [CLASS_CORE[c]  for c in blended],
        'bloom': [CLASS_BLOOM[c] for c in blended],
        'cw':    [CLASS_CORE_WIDTH[c]  for c in blended],
        'bw':    [CLASS_BLOOM_WIDTH[c] for c in blended],
        'label': pd.Timestamp(ts_str).strftime('%A %b %d %Y  %H:%M'),
    }
    print(f'  {ts_str}')

print('All hours precomputed')

Precomputing 24 hours …
Road segments : 9,916
  2016-03-19 23:00:00
  2016-03-20 00:00:00
  2016-03-20 01:00:00
  2016-03-20 02:00:00
  2016-03-20 03:00:00
  2016-03-20 04:00:00
  2016-03-20 05:00:00
  2016-03-20 06:00:00
  2016-03-20 07:00:00
  2016-03-20 08:00:00
  2016-03-20 09:00:00
  2016-03-20 10:00:00
  2016-03-20 11:00:00
  2016-03-20 12:00:00
  2016-03-20 13:00:00
  2016-03-20 14:00:00
  2016-03-20 15:00:00
  2016-03-20 16:00:00
  2016-03-20 17:00:00
  2016-03-20 18:00:00
  2016-03-20 19:00:00
  2016-03-20 20:00:00
  2016-03-20 21:00:00
  2016-03-20 22:00:00
All hours precomputed


In [ ]:
INIT_LAT, INIT_LON = 40.754, -73.978
INIT_ZOOM, INIT_PITCH, INIT_BEARING = 12, 55, -20

timestamps_js = _json.dumps(TIMELINE_HOURS)
road_paths_js = _json.dumps(road_paths)
timeline_js   = _json.dumps(timeline_data)

html = """<!DOCTYPE html>
<html>
<head>
<meta charset='utf-8'/>
<title>DeepDispatch — Revenue Heatmap</title>
<script src='https://unpkg.com/deck.gl@8.9.35/dist.min.js'></script>
<link href='https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css' rel='stylesheet'/>
<script src='https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js'></script>
<style>
*{margin:0;padding:0;box-sizing:border-box;}
body{background:#0d0d0d;font-family:'Segoe UI',sans-serif;overflow:hidden;}
#map{width:100vw;height:100vh;}
#ui{
  position:absolute;bottom:32px;left:50%;transform:translateX(-50%);
  background:rgba(10,10,20,0.88);border:1px solid rgba(255,255,255,0.1);
  border-radius:16px;padding:18px 28px;min-width:560px;
  backdrop-filter:blur(12px);box-shadow:0 8px 32px rgba(0,0,0,0.6);
}
#timestamp-label{color:#fff;font-size:15px;font-weight:600;text-align:center;margin-bottom:12px;letter-spacing:0.03em;}
#slider{width:100%;accent-color:#ff3c50;cursor:pointer;}
#tick-row{display:flex;justify-content:space-between;margin-top:6px;}
.tick{color:rgba(255,255,255,0.35);font-size:10px;text-align:center;}
#btn-row{display:flex;justify-content:center;gap:10px;margin-top:14px;}
button{background:rgba(255,255,255,0.08);border:1px solid rgba(255,255,255,0.15);color:#fff;padding:6px 16px;border-radius:8px;cursor:pointer;font-size:13px;transition:background 0.2s;}
button:hover{background:rgba(255,255,255,0.18);}
button.active{background:rgba(255,60,80,0.4);border-color:#ff3c50;}
#legend{
  position:absolute;top:24px;right:24px;
  background:rgba(10,10,20,0.88);border:1px solid rgba(255,255,255,0.1);
  border-radius:12px;padding:14px 18px;backdrop-filter:blur(12px);
}
#legend h4{color:#fff;font-size:12px;margin-bottom:10px;letter-spacing:0.06em;text-transform:uppercase;}
.legend-row{display:flex;align-items:center;gap:10px;margin-bottom:6px;}
.legend-dot{width:10px;height:10px;border-radius:50%;flex-shrink:0;}
.legend-lbl{color:rgba(255,255,255,0.7);font-size:12px;}
#loading-overlay{
  position:absolute;top:0;left:0;width:100%;height:100%;
  background:#0d0d0d;display:flex;flex-direction:column;align-items:center;justify-content:center;
  color:#fff;font-size:18px;z-index:999;transition:opacity 0.6s;
}
#loading-overlay p{margin-top:12px;font-size:13px;color:rgba(255,255,255,0.45);}
</style>
</head>
<body>
<div id='loading-overlay'>
  <span>Initializing map…</span>
  <p>Loading road geometry for 24 hours</p>
</div>
<div id='map'></div>
<div id='ui'>
  <div id='timestamp-label'>—</div>
  <input id='slider' type='range' min='0' max='23' step='0.01' value='0'/>
  <div id='tick-row'></div>
  <div id='btn-row'>
    <button id='btn-play'>▶ Play</button>
    <button id='btn-3d' class='active'>3D View</button>
    <button id='btn-flat'>Flat View</button>
    <button id='btn-reset'>⟳ Reset</button>
  </div>
</div>
<div id='legend'>
  <h4>Revenue Class</h4>
  <div class='legend-row'><div class='legend-dot' style='background:rgb(80,130,255)'></div><span class='legend-lbl'>Quiet</span></div>
  <div class='legend-row'><div class='legend-dot' style='background:rgb(0,230,200)'></div><span class='legend-lbl'>Steady</span></div>
  <div class='legend-row'><div class='legend-dot' style='background:rgb(100,255,100)'></div><span class='legend-lbl'>Busy</span></div>
  <div class='legend-row'><div class='legend-dot' style='background:rgb(255,210,0)'></div><span class='legend-lbl'>High Value</span></div>
  <div class='legend-row'><div class='legend-dot' style='background:rgb(255,60,80)'></div><span class='legend-lbl'>Surge</span></div>
</div>
<script>
const TIMESTAMPS = """ + timestamps_js + """;
const ROAD_PATHS = """ + road_paths_js + """;
const TIMELINE   = """ + timeline_js + """;

const tickRow = document.getElementById('tick-row');
TIMESTAMPS.forEach((ts, i) => {
  const el = document.createElement('span');
  el.className = 'tick';
  const d = new Date(ts);
  el.textContent = (i % 4 === 0) ? d.toLocaleTimeString([], {hour:'2-digit',minute:'2-digit'}) : '';
  tickRow.appendChild(el);
});

const {DeckGL, PathLayer} = deck;

const deckgl = new DeckGL({
  container: 'map',
  mapStyle: 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
  initialViewState: {
    latitude:  """ + str(INIT_LAT) + """,
    longitude: """ + str(INIT_LON) + """,
    zoom:      """ + str(INIT_ZOOM) + """,
    pitch:     """ + str(INIT_PITCH) + """,
    bearing:   """ + str(INIT_BEARING) + """,
  },
  controller: true,
  layers: [],
  onLoad: () => {
    const ov = document.getElementById('loading-overlay');
    ov.style.opacity = '0';
    setTimeout(() => ov.style.display = 'none', 600);
    setTimeout(() => renderHour(0), 100);
  }
});

const ROWS = TIMESTAMPS.map(ts => {
  const d = TIMELINE[ts];
  return ROAD_PATHS.map((path, i) => ({
    path,
    core_color:  d.core[i],
    bloom_color: d.bloom[i],
    core_width:  d.cw[i],
    bloom_width: d.bw[i],
  }));
});

function makeLayers(rows) {
  return [
    new PathLayer({
      id: 'bloom', data: rows,
      getPath:  r => r.path,
      getColor: r => r.bloom_color,
      getWidth: r => r.bloom_width,
      widthMinPixels: 1, rounded: true, pickable: false,
    }),
    new PathLayer({
      id: 'core', data: rows,
      getPath:  r => r.path,
      getColor: r => r.core_color,
      getWidth: r => r.core_width,
      widthMinPixels: 1, rounded: true, pickable: true, autoHighlight: true,
    }),
  ];
}

const slider  = document.getElementById('slider');
const label   = document.getElementById('timestamp-label');
const btnPlay = document.getElementById('btn-play');

function renderHour(idx) {
  deckgl.setProps({ layers: makeLayers(ROWS[idx]) });
  label.textContent = TIMELINE[TIMESTAMPS[idx]].label;
  slider.value = idx;
}

slider.addEventListener('input', () => {
  if (playTimer) { cancelAnimationFrame(playTimer); playTimer = null; animStart = null; btnPlay.textContent = '▶ Play'; }
  renderHour(Math.floor(+slider.value));
});

let playTimer = null;
let animStart = null;
const STEP_MS = 700;

function lerpColor(a, b, t) {
  return [0,1,2,3].map(i => Math.round(a[i] + (b[i] - a[i]) * t));
}

function animLoop(ts) {
  if (!animStart) animStart = ts;
  const total = STEP_MS * TIMESTAMPS.length;
  const frac  = ((ts - animStart) % total) / STEP_MS;
  const idxA  = Math.floor(frac) % TIMESTAMPS.length;
  const idxB  = (idxA + 1) % TIMESTAMPS.length;
  const t     = frac - Math.floor(frac);

  const rowsA = ROWS[idxA];
  const rowsB = ROWS[idxB];

  const blended = rowsA.map((r, i) => ({
    path:        r.path,
    core_color:  lerpColor(r.core_color,  rowsB[i].core_color,  t),
    bloom_color: lerpColor(r.bloom_color, rowsB[i].bloom_color, t),
    core_width:  r.core_width,
    bloom_width: r.bloom_width,
  }));

  deckgl.setProps({ layers: makeLayers(blended) });
  label.textContent = TIMELINE[TIMESTAMPS[idxA]].label;
  slider.value = frac % TIMESTAMPS.length;

  playTimer = requestAnimationFrame(animLoop);
}

btnPlay.addEventListener('click', function() {
  if (playTimer) {
    cancelAnimationFrame(playTimer);
    playTimer = null;
    animStart = null;
    this.textContent = '▶ Play';
  } else {
    this.textContent = '⏸ Pause';
    const startOffset = +slider.value * STEP_MS;
    animStart = performance.now() - startOffset;
    playTimer = requestAnimationFrame(animLoop);
  }
});

document.getElementById('btn-3d').addEventListener('click', function() {
  deckgl.setProps({ initialViewState: { latitude:""" + str(INIT_LAT) + """, longitude:""" + str(INIT_LON) + """, zoom:""" + str(INIT_ZOOM) + """, pitch:55, bearing:-20, transitionDuration:800 } });
  this.classList.add('active');
  document.getElementById('btn-flat').classList.remove('active');
});

document.getElementById('btn-flat').addEventListener('click', function() {
  deckgl.setProps({ initialViewState: { latitude:""" + str(INIT_LAT) + """, longitude:""" + str(INIT_LON) + """, zoom:""" + str(INIT_ZOOM) + """, pitch:0, bearing:0, transitionDuration:800 } });
  this.classList.add('active');
  document.getElementById('btn-3d').classList.remove('active');
});

document.getElementById('btn-reset').addEventListener('click', () => {
  deckgl.setProps({ initialViewState: { latitude:""" + str(INIT_LAT) + """, longitude:""" + str(INIT_LON) + """, zoom:""" + str(INIT_ZOOM) + """, pitch:""" + str(INIT_PITCH) + """, bearing:""" + str(INIT_BEARING) + """, transitionDuration:800 } });
});
</script>
</body>
</html>"""

out_html = os.path.join(OUTPUT_PATH, 'deepdispatch_timeline.html')
with open(out_html, 'w') as f:
    f.write(html)

size_mb = os.path.getsize(out_html) / 1e6
print(f'Saved → {out_html}')
print(f'File size : {size_mb:.1f} MB')
print('Open in browser. Play animates smoothly, slider glides continuously.')

Saved → /content/drive/MyDrive/SU Works/CPSC_5310_Project/notebooks/final_heatmap/deepdispatch_timeline.html
File size : 13.1 MB
Open in browser. Play animates smoothly, slider glides continuously.


In [ ]:
rows = []
for ts_str in TIMELINE_HOURS:
    d = timeline_data[ts_str]
    for i, path in enumerate(road_paths):
        rows.append({
            'timestamp':     ts_str,
            'revenue_class': int(np.clip(np.round(
                                 (np.array(d['core'][i]) * 0).sum()), 1, 5)),  # use blended class
            'label':         d['label'],
            'lon_start':     path[0][0],
            'lat_start':     path[0][1],
            'lon_end':       path[-1][0],
            'lat_end':       path[-1][1],
        })

df_export = pd.DataFrame(rows)
df_export.to_csv(os.path.join(OUTPUT_PATH, 'road_heatmap_24h.csv'), index=False)
print(f'Exported {len(df_export):,} rows')

Exported 237,984 rows
